## `nz_polybias.ipynb`
-------------

**polynomial photometric bias law.**

In [1]:
import numpy as np
import importlib
import json
import matplotlib.pyplot as plt
import pandas as pd
import scipy.interpolate as interp

from datetime import datetime
from pathlib import Path

import src.statistics.biasfit as biasfit
import src.statistics.cosmotools as ct
import src.statistics.corrfiles as cf
import src.statistics.inference as inference
import src.statistics.systematics as sy
import src.analysis.plots as plots

## autoreload
importlib.reload(biasfit)
importlib.reload(ct)
importlib.reload(inference)
importlib.reload(sy)

ROOT = cf.get_base_dir()
CORR_ROOT = ROOT / "src" / "statistics" / "outputs"

FIGURES_ROOT = ROOT / "paper" / "figures" / "systematics"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
pm = plots.PlotManager(root=FIGURES_ROOT, overwrite=True)

KeyboardInterrupt: 

In [ ]:
scale_cut = [0.3, 3]
version = "v_1p1"

tomo_to_tracer = sy.TOMO_TO_TRACER
patches = sy.PATCHES
tag = sy.scale_cut_tag(scale_cut)

FID_DIR = ROOT / "results" / "distributions" / f"{tag}_{version}"
OUT_DIR = sy.variant_dir(ROOT, "polybias", scale_cut, version)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Scale cut      : {scale_cut[0]} - {scale_cut[1]} Mpc/h, version {version}")
print(f"Fiducial input : {FID_DIR}")
print(f"Output         : {OUT_DIR}")

In [ ]:
fit = biasfit.refit_photoz_bias(scale_cut=scale_cut, version=version, overwrite=False)

for mode in biasfit.MODES:
    p = fit[f"{mode}/params"]
    c = fit[f"{mode}/cov"]
    print(f"--- {mode} ---")
    for nm, v, e in zip("abg", p, np.sqrt(np.diag(c))):
        print(f"    {nm} = {v: .8f} +- {e:.8f}")
    print(f"    chi2/dof = {float(fit[f'{mode}/chi2']):.1f}/{int(fit[f'{mode}/dof'])}")
    print(f"    correlation matrix:\n{biasfit.correlation_matrix(c)}\n")

In [ ]:
z_fit = fit["z"]
vals, errs = fit["vals"], fit["errs"]
zv_plot = np.linspace(0.05, 2.4, 200)

models = {
    (mode, use_cov): biasfit.PhotoBiasModel(
        mode, fit[f"{mode}/params"], fit[f"{mode}/cov"], use_covariance=use_cov
    )
    for mode in biasfit.MODES
    for use_cov in (True, False)
}

with pm.make_plot(
    f"polybias_error_check_{tag}_{version}", figsize=(7, 9), nrows=2, show=True
) as (fig, axs):
    styles = {"powerlaw": ("tab:blue", "-"), "polynomial": ("crimson", "--")}

    for mode in biasfit.MODES:
        color, ls = styles[mode]
        m_full, m_diag = models[(mode, True)], models[(mode, False)]
        f = m_full.value(zv_plot)
        axs[0].plot(zv_plot, f, color=color, ls=ls, lw=2, label=f"{mode}")
        axs[0].fill_between(
            zv_plot, f - m_diag.sigma(zv_plot), f + m_diag.sigma(zv_plot),
            color=color, alpha=0.15, label=f"{mode}, diagonal only",
        )
        axs[0].fill_between(
            zv_plot, f - m_full.sigma(zv_plot), f + m_full.sigma(zv_plot),
            color=color, alpha=0.55, label=f"{mode}, full covariance",
        )
        axs[1].plot(
            zv_plot, 100 * m_full.rel_sigma(zv_plot),
            color=color, ls=ls, lw=2, label=f"{mode}, full covariance",
        )
        axs[1].plot(
            zv_plot, 100 * m_diag.rel_sigma(zv_plot),
            color=color, ls=":", lw=2, alpha=0.8, label=f"{mode}, diagonal only",
        )

    axs[0].errorbar(
        z_fit, vals, errs, fmt="d", ms=3.5, capsize=3, color="purple",
        label=r"$\sqrt{\bar{\omega}_{pp}^{\mathrm{true}}}$", zorder=5,
    )
    axs[0].axvspan(
        z_fit.min(), z_fit.max(), color="gray", alpha=0.08, zorder=0, label="fit range"
    )
    axs[0].set_xlabel(r"Redshift ($z$)")
    axs[0].set_ylabel(r"$\sqrt{\bar{\omega}_{pp}}$")
    axs[0].set_ylim(0.0, 1.0)
    axs[0].grid(True, alpha=0.3)
    axs[0].legend(fontsize=9, ncols=2, loc="upper left")

    axs[1].axvspan(z_fit.min(), z_fit.max(), color="gray", alpha=0.08, zorder=0)
    axs[1].set_yscale("log")
    axs[1].set_xlabel(r"Redshift ($z$)")
    axs[1].set_ylabel(r"$\sigma_f / f$  [%]")
    axs[1].grid(True, alpha=0.3, which="both")
    axs[1].legend(fontsize=9)
    axs[1].set_title("Relative uncertainty on the bias correction")
    fig.tight_layout()

In [ ]:
zs = np.array([0.05, 0.3, 0.6, 0.9, 1.2, 1.6, 2.0, 2.4])
tblrows = []
for mode in biasfit.MODES:
    for use_cov in (True, False):
        m = models[(mode, use_cov)]
        for z in zs:
            tblrows.append(
                {
                    "mode": mode,
                    "covariance": "full" if use_cov else "diagonal",
                    "z": z,
                    "f(z)": m.value(z),
                    "sigma_f/f [%]": 100 * m.rel_sigma(z),
                }
            )
check = pd.DataFrame(tblrows).pivot_table(
    index="z", columns=["mode", "covariance"], values="sigma_f/f [%]"
)

In [ ]:
BIAS_MODE = "polynomial"

params = fit[f"{BIAS_MODE}/params"]
cov = fit[f"{BIAS_MODE}/cov"]
chi2, dof = float(fit[f"{BIAS_MODE}/chi2"]), int(fit[f"{BIAS_MODE}/dof"])

bias_model = biasfit.PhotoBiasModel(BIAS_MODE, params, cov, use_covariance=True)

In [ ]:
## precompute w_dm (same grid as nz.ipynb)
wdm_interpolator = sy.precompute_wdm_interpolator(scale_cut)
wdm_cache = sy.WdmCache(scale_cut)

In [ ]:
# estimated runtime on NERSC ~5 minutes
rows = []
for tomo in sy.TOMO_BINS:
    stem = sy.stem_for_tomo(tomo)
    path_dictionary = sy.build_path_dictionary(CORR_ROOT, stem, version)
    fr = cf.CorrFileReader(path_dictionary["DESIxHSC"])

    for tracer in tomo_to_tracer[tomo]:
        zbins = fr.get_bins(tracer)
        zvals = (zbins[:-1] + zbins[1:]) / 2

        meas = inference.full_npz_tomo(
            path_dictionary=path_dictionary,
            do_phot_correction=True,
            do_spec_correction=True,
            scale_cuts=scale_cut,
            tomo_bin=tomo,
            tracer=tracer,
            which_patches=patches,
            precomp_wdm=wdm_interpolator,
            bias_model=bias_model,
        )
        assert not np.any(meas[1] <= 0) and not np.any(np.isnan(meas[1]))
        assert len(meas[0]) == len(zvals)

        npz_mag, npz_mag_err = ct.solve_magnification(
            meas=meas,
            tracer=tracer,
            tomo_bin=tomo,
            scale_cut=scale_cut,
            zvalues=zvals,
            w_dm_values=wdm_cache.get(zvals),
            bias_mode=BIAS_MODE,
        )
        print(f"DR {stem}, tomo {tomo}, {tracer}: {len(meas[0])} redshift bins")

        for j, z in enumerate(zvals):
            rows.append(
                {
                    "tomo_bin": tomo,
                    "tracer": tracer,
                    "redshift": z,
                    "npz_bs_bp": meas[0][j],
                    "npz_bs_bp_err": meas[1][j],
                    "npz_bs_bp_mag": npz_mag[j],
                    "npz_bs_bp_mag_err": npz_mag_err[j],
                }
            )

df_poly = pd.DataFrame(rows)
df_poly.head()

In [ ]:
fid = pd.read_parquet(FID_DIR / f"nz_res_{tag}_{version}.parquet")
carry = fid[["tomo_bin", "tracer", "redshift", "npz_cross", "npz_cross_err"]].copy()
carry["npz_bs"] = fid["npz_bs"]
carry["npz_bs_err"] = fid["npz_bs_err"]

df = carry.merge(df_poly, on=["tomo_bin", "tracer", "redshift"], how="outer")
n_matched = df["npz_bs_bp"].notna().sum()
assert n_matched == len(df_poly), (
    f"{len(df_poly) - n_matched} recomputed rows did not match a fiducial redshift; "
    "check that the fiducial parquet uses the same binning."
)

df.to_parquet(OUT_DIR / f"nz_res_{tag}_{version}.parquet", index=False)
print(f"{len(df)} rows -> {OUT_DIR / f'nz_res_{tag}_{version}.parquet'}")
df.head()

In [ ]:
merged = sy.merge_over_tracers(df, names=sy.NAMES)
norm = sy.normalize_merged(merged, names=sy.NAMES)

np.savez_compressed(OUT_DIR / f"merged_res_{tag}_{version}.npz", **merged)
np.savez_compressed(OUT_DIR / f"merged_res_norm_{tag}_{version}.npz", **norm)

metadata = {
    "study": "polybias",
    "description": (
        "Photometric galaxy-bias correction replaced by a degree-2 polynomial "
        "alpha(1+z)^2 + beta(1+z) + gamma, with the full fit covariance propagated "
        "into the n(z) error bars. npz_cross and npz_bs are carried over unchanged "
        "from the fiducial run since they do not use the photometric bias law."
    ),
    "scale_cuts": scale_cut,
    "version": version,
    "bias_mode": BIAS_MODE,
    "use_covariance": True,
    "bias_params": params.tolist(),
    "bias_cov": np.asarray(cov).tolist(),
    "bias_chi2": chi2,
    "bias_dof": dof,
    "patches": patches,
    "tracers_by_tomo": {str(k): v for k, v in tomo_to_tracer.items()},
    "fiducial_input": str(FID_DIR / f"nz_res_{tag}_{version}.parquet"),
    "creation_date": datetime.now().isoformat(),
}
with open(OUT_DIR / f"polybias_metadata_{tag}_{version}.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved merged + normalized n(z) and metadata to {OUT_DIR}")

### 3. Effect on the measured error bars (pre-spline)

In [ ]:
fid_norm = np.load(FID_DIR / f"merged_res_norm_{tag}_{version}.npz")
bin_colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:red", 4: "tab:cyan"}

with pm.make_plot(
    f"polybias_points_{tag}_{version}", figsize=(11, 7), nrows=2, ncols=2, show=True
) as (fig, axs):
    for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
        key = f"{tomo}/npz_bs_bp_mag"
        zf, nf, ef = (fid_norm[f"{key}_z"], fid_norm[key], fid_norm[f"{key}_err"])
        zp, np_, ep = (norm[f"{key}_z"], norm[key], norm[f"{key}_err"])

        ax.errorbar(zf, nf, ef, fmt="D", ms=4, capsize=3, color="0.35", label="powerlaw")
        ax.errorbar(
            zp + 0.004, np_, ep, fmt="s", ms=4, capsize=3,
            color=bin_colors[tomo], label="polynomial",
        )
        ax.axhline(0, color="k", ls="--", lw=1)
        ratio = np.median(ep / ef)
        ax.set_title(f"Bin {tomo} (median $\\sigma$ ratio {ratio:.3f})")
        ax.set_xlabel("Redshift")
        ax.set_ylabel("n(z)")
        ax.grid(True, alpha=0.3)
        if tomo == 1:
            ax.legend(fontsize=9)
    fig.suptitle("Magnification-corrected n(z): power-law vs polynomial bias correction")
    fig.tight_layout()

In [ ]:
for name in ("npz_bs_bp", "npz_bs_bp_mag"):
    print(f"--- {name} ---")
    for tomo in sy.TOMO_BINS:
        key = f"{tomo}/{name}"
        ef = fid_norm[f"{key}_err"]
        ep = norm[f"{key}_err"]
        shift = (norm[key] - fid_norm[key]) / ef
        print(
            f"  bin {tomo}: sigma ratio median {np.median(ep / ef):.3f} "
            f"[{np.min(ep / ef):.3f}, {np.max(ep / ef):.3f}] | "
            f"shift median {np.median(shift):+.3f}, max |shift| "
            f"{np.abs(shift).max():.3f} sigma_fid"
        )